In this notebook we comapre k-means clustering on CLIP embeddings of a dataset with k-means clustering on the raw pixel features of the images in the Pattern dataset. <br>
We hypothesize that in the CLIP space, the clusters formed from the CLIP embeddings will represent semantic similarities in the patterns, e.g., patterns that arise in similar materials, similar objects <br>
On the other hand, we expect the clusters formed from the raw pixel vectors to be grouped based on more high-level similarities in the features of the images, e.g., similiarity in the shapes in the pattern no matter the mediums in which the patterns occur, colors, etc. <br> 
(starter code from Dhanashree (Birds dataset) with new code, techniques, and observations written by Megha and Naomi for the Patterns dataset.)

In [1]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from PIL import Image
import random

## Clustering of CLIP embeddings

In [2]:
# Load patterns_emb. 
patterns_emb = np.load('/home/d4205992-6e79-432d-bc8b-0c0503a40624/COGS118B_FinalProject/patterns_emb.npy')

In [ ]:
# Plotting Elbow curve to find a god estimate for number of clusters. 
inertias = [] # Sum of squared distances of each data point to its closest cluster center.
K_range = range(2, 40)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=0)
    kmeans.fit(patterns_emb)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, inertias, marker="o")
plt.title("Elbow Method (CLIP Embeddings — Patterns)")
plt.xlabel("Number of clusters K")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

The graph does not show a clear elbow. So we will use another method to determine the best number of clusters, called the Silhouette Score.

## SILHOUETTE SCORE

In [ ]:
# embeddings: your (N, D) CLIP embedding matrix
# Ks: the range of cluster numbers
Ks = range(2, 41)
sil_scores = []

for k in Ks:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(patterns_emb)
    score = silhouette_score(patterns_emb, labels)
    sil_scores.append(score)
    print(f"K={k}, silhouette={score:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(Ks, sil_scores, marker='o')
plt.xlabel("Number of clusters K")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for K-means on CLIP Embeddings")
plt.grid(True)
plt.show()
best_k = Ks[np.argmax(sil_scores)]
best_score = max(sil_scores)

print(f"\n Best K = {best_k} with score = {best_score:.4f}")

In [23]:
best_k_clip = 36

# Train k-means on CLIP embeddings.
kmeans_clip = KMeans(n_clusters=best_k_clip, random_state=0)
clip_clusters = kmeans_clip.fit_predict(patterns_emb)

## Clustering for Pixel space 

In [24]:
def get_all_images_pathlib(root_folder):
    root_path = Path(root_folder)
    images = []
    
    for file_path in root_path.rglob('*'):
        if file_path.suffix.lower() == '.jpg':
            images.append(str(file_path))
    
    return images

def load_pixel_vectors(image_paths, size=(128,128)):
    pixel_vectors = []
    for p in image_paths:
        img = Image.open(p).convert("RGB").resize(size)
        arr = np.array(img).flatten()
        pixel_vectors.append(arr) # Add the vector for each image.
    return np.array(pixel_vectors)

In [25]:
base_dir = '/home/mekumar/'
patterns_folder = f'{base_dir}COGS118B_FinalProject-main/Patterns'
patterns = get_all_images_pathlib(patterns_folder)
print(f"Found {len(patterns)} images")

pixel_data = load_pixel_vectors(patterns) # This contains the pixel data for the Patterns images. 
print("Pixel data shape:", pixel_data.shape)

# Reduce the dimentionality of the pixel features using PCA.
# WHY IS THIS IMPORTANT? -> Makes the clusters more stable.
# The sklearn documentation also mentions that for very high dimensionality, k-means performance is impacted. 
# We do not resize to a very small size to preserve the details of the image. 

pca_pixels = PCA(n_components = 50)   
pixel_pcs = pca_pixels.fit_transform(pixel_data)

print("PCA pixel feature shape:", pixel_pcs.shape)

Found 1392 images


In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
K_range = range(2, 30)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(pixel_pcs)
    score = silhouette_score(pixel_pcs), labels)
    sil_scores.append(score)

plt.figure(figsize=(8,5))
plt.plot(K_range, sil_scores, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Method on Pixel Embeddings")
plt.grid(True)
plt.show()

best_k = Ks[np.argmax(sil_scores)]
best_score = max(sil_scores)

print(f"\n Best K = {best_k} with score = {best_score:.4f}")

In [ ]:
best_k_clip = 28

# Train k-means on CLIP embeddings.
kmeans_clip = KMeans(n_clusters=best_k_clip, random_state=0)
pixel_clusters = kmeans_clip.fit_predict(patterns_emb)

In [ ]:
# We would like to see how the pattern images were grouped differently.
# To do this we define a function which lets us display "n" number of images from a cluster of or choice.
# We refer to a cluster by its number. 
def show_cluster_samples(image_paths, labels, cluster_id, n=28):
    
    idx = [i for i in range(len(labels)) if labels[i] == cluster_id] # Get the indexes of images in a cluster. 
    # Here label is the cluster number assigned to an image. 
    # cluster_id is the cluster we want to look into. 
    # This works because CLIP embeddings retain the original ordering of images. So we can correlate this to the original images. 
    
    if len(idx) == 0: 
        print("cluster empty") # If we try to check a cluster that doesn't exist.
        return
    
    sample = random.sample(idx, min(n, len(idx))) # Randomly selecting indexes of images in a cluster.

    # Set style specifics for the plot of images. 
    cols = 4  
    rows = (len(sample) + cols - 1) // cols 
    plt.figure(figsize=(10, 10))
    
    for j, i in enumerate(sample):
        img = Image.open(image_paths[i]).convert("RGB")
        ax = plt.subplot(rows, cols, j+1)
        plt.imshow(img) # Display the images. 
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [30]:
# example clusters
clip_k = 36      # Pick any valid cluster number from clip clusters.
pixel_k = 20    # Pick any valid cluster number from pixel clusters.

print("CLIP cluster:")
show_cluster_samples(patterns, clip_clusters, clip_k, n=15)

print("Pixel cluster:")
show_cluster_samples(patterns, pixel_clusters, pixel_k, n=15)

In [ ]:
# example clusters
clip_k = 10      # Pick any valid cluster number from clip clusters (1 to 20).
pixel_k = 9    # Pick any valid cluster number from pixel clusters (1 to 16).

print("CLIP cluster:")
show_cluster_samples(patterns, clip_clusters, clip_k, n=15)

print("Pixel cluster:")
show_cluster_samples(patterns, pixel_clusters, pixel_k, n=15)


In [ ]:
# example clusters
clip_k = 5      # Pick any valid cluster number from clip clusters.
pixel_k = 7    # Pick any valid cluster number from pixel clusters.

print("CLIP cluster:")
show_cluster_samples(patterns, clip_clusters, clip_k, n=15)

print("Pixel cluster:")
show_cluster_samples(patterns, pixel_clusters, pixel_k, n=15)


In [ ]:
# example clusters
clip_k = 18     # Pick any valid cluster number from clip clusters (1 to 20).
pixel_k = 3    # Pick any valid cluster number from pixel clusters (1 to 10).

print("CLIP cluster:")
show_cluster_samples(patterns, clip_clusters, clip_k, n=15)

print("Pixel cluster:")
show_cluster_samples(patterns, pixel_clusters, pixel_k, n=15)


In [ ]:
# example clusters
clip_k = 7    # Pick any valid cluster number from clip clusters (1 to 20).
pixel_k = 15    # Pick any valid cluster number from pixel clusters (1 to 10).

print("CLIP cluster:")
show_cluster_samples(patterns, clip_clusters, clip_k, n=15)

print("Pixel cluster:")
show_cluster_samples(patterns, pixel_clusters, pixel_k, n=15)


## Observations

The CLIP-based clusters tend to group together textures and surfaces that share a similar material or structural identity, such as layered rock formations, organic growth patterns, woven or fibrous surfaces, and regularly patterned materials like meshes or scales. These groupings often include a mix of colors and contexts, but still feel visually and conceptually coherent because they reflect recognizable types of natural or manufactured textures. In contrast, the pixel-based clusters appear to be driven primarily by shared visual qualities such as darkness or lightness, overall color tone, and the density of small, repeated marks. As a result, surfaces like hair, grass, fabric, and rough ground are sometimes grouped together simply because they share similar visual densities or color profiles, even though they are semantically unrelated. Overall, the CLIP clusters emphasize material and structural similarity, while the pixel clusters emphasize purely visual resemblance at the surface level.